# wordle-0.2b — benchmark against real LLM APIs

Same secret words, same rules, everyone gets 6 guesses. Chat models get the conversation history and must reply with a single valid word; the local model and the teacher play through the same game engine. Runs headless too: see `docs/benchmarks.md` for the CLI version.

API keys come from environment variables (or a `.env` file in the repo root, gitignored). No keys, no problem: the notebook still runs the local contestants.

In [ ]:
import sys, os
from pathlib import Path

REPO = Path.cwd().parent if (Path.cwd() / "src").exists() else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO))

if (REPO / ".env").exists():
    for line in (REPO / ".env").read_text().splitlines():
        if "=" in line and not line.strip().startswith("#"):
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip())

import numpy as np
from wordle02b import Vocabulary, load_word_lists

vocab = Vocabulary(*load_word_lists())
print(f"{vocab.V} words, {len(vocab.answers)} answers")

## Contestants

The notebook picks everyone it can: `baseline` always, `local0.2b` if a checkpoint exists, and any API models whose keys are set. Add your own entries in the list below — any model id you have a key for: `openai/gpt-4.1`, `openrouter/anthropic/claude-sonnet-4-5`, `local/your-model` (with `LOCAL_API_BASE` set) and so on.

In [ ]:
entries = ["baseline"]

cands = sorted(Path("checkpoints").glob("*.pt")) if Path("checkpoints").exists() else []
CKPT = str(cands[-1]) if cands else None
if CKPT:
    entries.append("local0.2b")

if os.environ.get("OPENAI_API_KEY"):   entries.append("openai/gpt-4o-mini")
if os.environ.get("ANTHROPIC_API_KEY"): entries.append("anthropic/claude-sonnet-4-5")
if os.environ.get("DEEPSEEK_API_KEY"): entries.append("deepseek/deepseek-chat")
if os.environ.get("GROQ_API_KEY"):     entries.append("groq/llama-3.3-70b-versatile")
if os.environ.get("GEMINI_API_KEY"):   entries.append("gemini/gemini-2.5-flash")

print("contestants:", entries)

In [ ]:
from benchmarks.api_benchmark import run_benchmark, summarize_rows

N_GAMES = 30  # games per contestant; costs cents for most models
rng = np.random.default_rng(42)
secrets = [str(rng.choice(vocab.answers)) for _ in range(N_GAMES)]

rows = run_benchmark(
    [tuple(e.split("/", 1)) if "/" in e else (e, e) for e in entries],
    secrets, vocab, checkpoint=CKPT,
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(summarize_rows(list(rows.items()))).sort_values("solve_rate", ascending=False)
display(df)

fig, ax = plt.subplots(figsize=(9, 4.2))
best = df["solve_rate"].max()
colors = ["#6aaa64" if v == best else "#c9b458" if v > 0.5 else "#787c7e" for v in df["solve_rate"]]
ax.barh(df["model"], df["solve_rate"] * 100, color=colors)
ax.set_xlabel("solve rate %"); ax.set_xlim(0, 105)
ax.set_title(f"Wordle solve rate, {N_GAMES} games each")
for i, v in enumerate(df["solve_rate"] * 100):
    ax.text(v + 1, i, f"{v:.1f}%", va="center", fontsize=9)
plt.tight_layout(); plt.show()

## Reading the results

- **baseline / teacher** — the algorithmic ceiling. ~99% on the official list.
- **local0.2b** — your model. Fully trained it should sit just below the teacher. That gap is the imitation tax.
- **API models** — frontier chat models typically land 60-95% on this protocol. They don't get a word list, they reason in natural language, and they occasionally hallucinate words or repeat guesses. They also cost money and take seconds per move, vs milliseconds locally.

Expectations are the point of this benchmark: a 0.2B model trained for 30 minutes beats frontier LLMs at this specific game, because the game is a closed, fully observable puzzle and the model was trained for exactly it. The LLMs win at everything where language understanding matters. Different tools, different jobs.

Cost notes: 30 games per model is a few cents on cheap models (gpt-4o-mini, deepseek-chat, gemini-flash) and a dollar or two on the expensive ones. The CSV in `results/` records tokens and estimated cost per model.